# 📄 pdf2md-ocr

> PDF 轉 Markdown，支援簡繁中文 OCR，大型 PDF 分批處理

[![GitHub](https://img.shields.io/badge/GitHub-pdf2md--ocr-black?logo=github)](https://github.com/YOUR_USERNAME/pdf2md-ocr)

**功能：**
- 自動偵測掃描版 / 文字版，混合 PDF 也能處理
- 每 100 頁分批，避免大檔（44MB+）記憶體爆炸
- 支援簡體 `chi_sim`、繁體 `chi_tra`、英文 `eng` 自由組合
- 輸出帶頁碼標記的標準 Markdown

## 1️⃣ 安裝依賴

In [ ]:
!apt-get update -qq
!apt-get install -y -qq \
    tesseract-ocr \
    tesseract-ocr-chi-sim \
    tesseract-ocr-chi-tra \
    poppler-utils

!pip install -q pytesseract pymupdf Pillow pypdf tqdm

print('✅ 安裝完成')

## 2️⃣ 下載轉換腳本

In [ ]:
# 從 GitHub 下載最新版本的轉換腳本
!wget -q https://raw.githubusercontent.com/YOUR_USERNAME/pdf2md-ocr/main/pdf2md.py
print('✅ 腳本下載完成')

## 3️⃣ 上傳 PDF

In [ ]:
from google.colab import files
import os

print('請選擇要上傳的 PDF 檔案...')
uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]
pdf_path = f'/content/{pdf_filename}'
size_mb = os.path.getsize(pdf_path) / 1e6

print(f'\n✅ 已上傳：{pdf_filename}（{size_mb:.1f} MB）')

## 4️⃣ 設定參數並執行

In [ ]:
# ── 可調整的參數 ──────────────────────────
LANG      = 'chi_sim+eng'  # chi_sim=簡體, chi_tra=繁體, eng=英文
DPI       = 200            # 150=快, 200=均衡, 300=高品質
CHUNK     = 100            # 每批頁數，記憶體不足可改 50
FORCE_OCR = False          # True=強制 OCR（忽略文字層）
# ──────────────────────────────────────────

output_md = pdf_path.replace('.pdf', '.md')

force_flag = '--force-ocr' if FORCE_OCR else ''
!python pdf2md.py "{pdf_path}" \
    --output "{output_md}" \
    --lang {LANG} \
    --dpi {DPI} \
    --chunk {CHUNK} \
    {force_flag}

## 5️⃣ 預覽結果

In [ ]:
with open(output_md, encoding='utf-8') as f:
    content = f.read()

size_kb = os.path.getsize(output_md) / 1024
print(f'📄 輸出：{os.path.basename(output_md)}')
print(f'   大小：{size_kb:.0f} KB | 字元：{len(content):,}')
print('\n' + '='*50)
print(content[:2000])
print('\n... (顯示前 2000 字元)')

## 6️⃣ 下載 Markdown 檔案

In [ ]:
from google.colab import files
files.download(output_md)
print(f'⬇️  正在下載：{os.path.basename(output_md)}')

## 🔧 進階：針對特定頁面重新 OCR

In [ ]:
# 若某幾頁識別效果不佳，可單獨以更高 DPI 重跑
import fitz, pytesseract, io
from PIL import Image

REPROCESS_PAGES = [1, 5]  # 要重跑的頁碼（1-indexed）
HIGH_DPI = 300

doc = fitz.open(pdf_path)
for pn in REPROCESS_PAGES:
    page = doc[pn - 1]
    mat = fitz.Matrix(HIGH_DPI/72, HIGH_DPI/72)
    pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    img = Image.open(io.BytesIO(pix.tobytes('png')))
    text = pytesseract.image_to_string(img, lang=LANG, config='--oem 3 --psm 6')
    print(f'\n=== 第 {pn} 頁（DPI={HIGH_DPI}）===')
    print(text[:500])
doc.close()

## 📌 參數速查

| 參數 | 預設 | 說明 |
|------|------|------|
| `LANG` | `chi_sim+eng` | OCR 語言組合 |
| `DPI` | `200` | 越高越準但越慢 |
| `CHUNK` | `100` | 每批頁數，OOM 時調小 |
| `FORCE_OCR` | `False` | 強制忽略文字層 |

**語言代碼：**  
`chi_sim`＝簡體中文 ｜ `chi_tra`＝繁體中文 ｜ `eng`＝英文  
可任意用 `+` 組合，例如 `chi_sim+chi_tra+eng`